# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaimAli0001/Flyrank-Internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 : Flags and Health Score

The paper reports that pages with more FlyRank optimization flags had higher Health Scores than pages with fewer flags.

My question is about **where the flags come from**. Are the flags independent observations, or are they created from the same underlying metrics that are also used in the Health Score?

If both use related information, the relationship may partly come from how FlyRank's diagnostic system is designed. This does not make the finding wrong, but it means the result should be interpreted as an observed relationship within the FlyRank system rather than independent evidence that flags cause better or worse page performance.

### Finding 2 : Content Age and Health Score

The paper reports that Health Score changes across content-age groups, with scores becoming lower again among older pages.

My question is about **validation and interpretation**. Does the comparison show that content age itself affects Health Score, or could other differences between newer and older pages explain the pattern?

For example, pages from different age groups may differ in topic, traffic, quality, or publishing period. The comparison therefore supports an observed relationship, but by itself does not prove that reaching a certain age causes Health Score to decline.

### Summary of the audit questions

| Paper finding | Methodology question |
|---|---|
| Flags and Health Score | Are the flags independent evidence, or are they derived from information also used in the Health Score? |
| Content Age and Health Score | Does the comparison support a causal age/freshness claim, or does it only show an observed association? |

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The Week-5 model used a client-grouped holdout and achieved strong observed Precision@K on the held-out clients.

For this audit, I use a stricter time-aware evaluation. The model is trained only on earlier month-to-next-month examples and evaluated on a later month-to-next-month example.

The training data uses:
- January → February CTR decline
- February → March CTR decline

The test data uses:
- March → April CTR decline

This preserves the same feature and target structure used in Week 5 while ensuring that the test outcome occurs after every outcome used for model training.

The comparison therefore asks whether the Week-5 ranking performance remains strong when evaluated in a more deployment-like future setting.

In [1]:
from dotenv import load_dotenv
import os
import duckdb
import pandas as pd

load_dotenv("../../.env")

token = os.getenv("HF_TOKEN")
print("Token loaded:", token is not None)

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

print("DuckDB connection ready.")

Token loaded: True
DuckDB connection ready.


In [2]:
months = [
    "2026-01",
    "2026-02",
    "2026-03",
    "2026-04",
    "2026-05",
    "2026-06",
]

month_availability = []

for m in months:
    result = con.sql(f"""
    SELECT
        '{m}' AS month,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(*) AS rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={m}/*.parquet'
    )
    """).df()

    month_availability.append(result)

month_availability = pd.concat(
    month_availability,
    ignore_index=True
)

month_availability

,month,first_date,last_date,rows
0,2026-01,2026-01-01,2026-01-31,7890817
1,2026-02,2026-02-01,2026-02-28,7355108
2,2026-03,2026-03-01,2026-03-31,9841378
3,2026-04,2026-04-01,2026-04-30,10424730
4,2026-05,2026-05-01,2026-05-31,11687376
5,2026-06,2026-06-01,2026-06-30,11694072


In [3]:
def build_monthly_model_frame(feature_month, outcome_month):
    """
    Build one month-to-next-month modeling frame.

    feature_month: month used for features, e.g. '2026-03'
    outcome_month: immediately following month used for target, e.g. '2026-04'
    """

    feature_rel = f"""
    read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={feature_month}/*.parquet'
    )
    """

    outcome_rel = f"""
    read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={outcome_month}/*.parquet'
    )
    """

    query = f"""
    WITH feature_daily AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position
        FROM {feature_rel}
        WHERE gsc_data_available IS TRUE
    ),

    feature_monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions) AS impressions,
            CASE
                WHEN SUM(gsc_impressions) > 0
                THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
                ELSE NULL
            END AS ctr,

            AVG(gsc_avg_position) AS avg_position,

            AVG(
                CASE
                    WHEN EXTRACT(DAY FROM report_date) <= 15
                    THEN gsc_impressions
                END
            ) AS first_half_impressions,

            AVG(
                CASE
                    WHEN EXTRACT(DAY FROM report_date) > 15
                    THEN gsc_impressions
                END
            ) AS second_half_impressions,

            SUM(
                CASE
                    WHEN EXTRACT(DAY FROM report_date) <= 15
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS first_half_clicks,

            SUM(
                CASE
                    WHEN EXTRACT(DAY FROM report_date) <= 15
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS first_half_total_impressions,

            SUM(
                CASE
                    WHEN EXTRACT(DAY FROM report_date) > 15
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS second_half_clicks,

            SUM(
                CASE
                    WHEN EXTRACT(DAY FROM report_date) > 15
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS second_half_total_impressions

        FROM feature_daily

        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    outcome_daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks
        FROM {outcome_rel}
        WHERE gsc_data_available IS TRUE
    ),

    outcome_monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS outcome_impressions,
            SUM(gsc_clicks) AS outcome_clicks,

            CASE
                WHEN SUM(gsc_impressions) > 0
                THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
                ELSE NULL
            END AS outcome_ctr

        FROM outcome_daily

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        f.client_hash_id,
        f.content_hash_id,

        f.impressions AS march_impressions,
        f.ctr AS march_ctr,
        f.avg_position AS march_avg_position,

        f.second_half_impressions
            - f.first_half_impressions
            AS impression_trend,

        CASE
            WHEN f.first_half_total_impressions > 0
             AND f.second_half_total_impressions > 0
            THEN
                (f.second_half_clicks * 1.0
                 / f.second_half_total_impressions)
                -
                (f.first_half_clicks * 1.0
                 / f.first_half_total_impressions)
            ELSE NULL
        END AS ctr_trend,

        o.outcome_ctr,

        CASE
            WHEN o.outcome_ctr < f.ctr THEN 1
            ELSE 0
        END AS ctr_decline

    FROM feature_monthly f

    INNER JOIN outcome_monthly o
        USING (client_hash_id, content_hash_id)

    WHERE f.ctr IS NOT NULL
      AND o.outcome_ctr IS NOT NULL
    """

    return con.sql(query).df()

In [9]:
jan_feb = build_monthly_model_frame("2026-01", "2026-02")
feb_mar = build_monthly_model_frame("2026-02", "2026-03")
mar_apr = build_monthly_model_frame("2026-03", "2026-04")

print("Jan → Feb:", jan_feb.shape)
print("Feb → Mar:", feb_mar.shape)
print("Mar → Apr:", mar_apr.shape)

Jan → Feb: (110867, 9)
Feb → Mar: (134238, 9)
Mar → Apr: (158549, 9)


In [10]:
FEATURES = [
    "march_impressions",
    "march_ctr",
    "march_avg_position",
    "impression_trend",
    "ctr_trend",
]

TARGET = "ctr_decline"

for name, df in [
    ("Jan → Feb", jan_feb),
    ("Feb → Mar", feb_mar),
    ("Mar → Apr", mar_apr),
]:
    print(f"\n{name}")
    print("Rows:", len(df))
    print("Missing:")
    print(df[FEATURES + [TARGET]].isna().sum())


Jan → Feb
Rows: 110867
Missing:
march_impressions         0
march_ctr                 0
march_avg_position        0
impression_trend      17743
ctr_trend             17743
ctr_decline               0
dtype: int64

Feb → Mar
Rows: 134238
Missing:
march_impressions         0
march_ctr                 0
march_avg_position        0
impression_trend      33325
ctr_trend             33325
ctr_decline               0
dtype: int64

Mar → Apr
Rows: 158549
Missing:
march_impressions         0
march_ctr                 0
march_avg_position        0
impression_trend      23291
ctr_trend             23291
ctr_decline               0
dtype: int64


In [11]:
TRAIN_FRAMES = [jan_feb, feb_mar]
TEST_FRAME = mar_apr

train_time = pd.concat(TRAIN_FRAMES, ignore_index=True)

X_time_train = train_time[FEATURES].copy()
y_time_train = train_time[TARGET].copy()

X_time_test = TEST_FRAME[FEATURES].copy()
y_time_test = TEST_FRAME[TARGET].copy()

print("Time-forward training rows:", len(X_time_train))
print("Time-forward test rows:", len(X_time_test))

print("\nTraining base rate:", y_time_train.mean())
print("Test base rate:", y_time_test.mean())

Time-forward training rows: 245105
Time-forward test rows: 158549

Training base rate: 0.2575957242814304
Test base rate: 0.3075137654605201


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

time_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000
    ))
])

time_model.fit(X_time_train, y_time_train)

time_prob = time_model.predict_proba(X_time_test)[:, 1]

print("Time-forward model trained.")
print("Predictions:", len(time_prob))

Time-forward model trained.
Predictions: 158549


In [15]:
def precision_at_k(df, score_column, k=20):
    ranked = df.sort_values(
        by=[score_column, "march_impressions", "march_ctr"],
        ascending=[False, False, True]
    )

    return ranked.head(k)["ctr_decline"].mean()

In [16]:
time_results = TEST_FRAME[
    [
        "client_hash_id",
        "content_hash_id",
        "march_impressions",
        "march_ctr",
        "march_avg_position",
        "impression_trend",
        "ctr_trend",
        "ctr_decline"
    ]
].copy()

time_results["model_probability"] = time_prob


time_precision = pd.DataFrame({
    "K": [20, 50, 100],
    "W05_grouped_split": [1.00, 1.00, 0.99],
    "W06_time_forward": [
        precision_at_k(time_results, "model_probability", 20),
        precision_at_k(time_results, "model_probability", 50),
        precision_at_k(time_results, "model_probability", 100)
    ],
    "base_rate": y_time_test.mean()
})

time_precision

,K,W05_grouped_split,W06_time_forward,base_rate
0,20,1.00,0.85,0.307514
1,50,1.00,0.86,0.307514
2,100,0.99,0.82,0.307514


The Week-5 values below are the previously observed results from the executed W05 notebook and are included as the before-condition for this audit. They are not recomputed or retuned in W06.

### Before vs. after result

The Week-5 client-grouped evaluation produced very high observed Precision@K: 1.00 at K=20, 1.00 at K=50, and 0.99 at K=100.

Under the stricter time forward evaluation, Precision@K decreased to 0.85, 0.86, and 0.82 respectively.

This performance drop suggests that the Week-5 result was optimistic relative to a future period deployment-style test. However, the time forward model still performed above the 30.75% test set base rate at all three review depths.

The audit therefore provides evidence of useful ranking signal, but the stronger Week 5 result should not be treated as evidence that the model will maintain the same performance on future periods.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final Week 5 feature set contains five March search performance features: impressions, CTR, average position, impression trend, and CTR trend.

The leakage audit checks three possible sources of leakage:

1. **Label-derived leakage:** whether a feature directly or indirectly contains information used to define `ctr_decline`.
2. **Future window leakage:** whether a feature contains information from April or any period after the prediction point.
3. **Decision derived leakage:** whether a feature is an existing FlyRank score or flag that already encodes a prior decision.

The prediction timeline is:

March 2026 features → prediction point → April 2026 CTR decline.

The model features are therefore constructed before the target window. Week 4 scores and action labels are kept outside the feature set and used only as the frozen baseline.

The remaining concern is that very strong model performance should still be treated cautiously and checked against the feature definitions and observed error patterns.

In [18]:
feature_audit = pd.DataFrame({
    "feature": FEATURES,
    "derived_from_april_target": ["No"] * len(FEATURES),
    "uses_future_april_data": ["No"] * len(FEATURES),
    "decision_derived": ["No"] * len(FEATURES)
})

feature_audit

,feature,derived_from_april_target,uses_future_april_data,decision_derived
0,march_impressions,No,No,No
1,march_ctr,No,No,No
2,march_avg_position,No,No,No
3,impression_trend,No,No,No
4,ctr_trend,No,No,No


In [19]:
print("Features used by the model:")
for feature in FEATURES:
    print("-", feature)

print("\nTarget:")
print(TARGET)

print("\nW04 decision fields excluded from features:")
print([
    "priority_score",
    "action_label",
    "reason_code"
])

Features used by the model:
- march_impressions
- march_ctr
- march_avg_position
- impression_trend
- ctr_trend

Target:
ctr_decline

W04 decision fields excluded from features:
['priority_score', 'action_label', 'reason_code']


In [20]:
timeline_audit = pd.DataFrame({
    "stage": [
        "Feature window",
        "Prediction point",
        "Target window"
    ],
    "period": [
        "March 2026",
        "After March 2026",
        "April 2026"
    ]
})

timeline_audit

,stage,period
0,Feature window,March 2026
1,Prediction point,After March 2026
2,Target window,April 2026


### Leakage audit conclusion

The final feature set contains only March 2026 search-performance signals, while `ctr_decline` is defined from the following April outcome window.

No obvious label derived leakage was identified in the five model features. The features do not use April outcome information, and no Week-4 decision outputs such as `priority_score`, `action_label`, or `reason_code` were used as model inputs.

The feature timeline therefore respects the prediction boundary: March information is available before the April outcome. These checks did not identify an obvious leakage source, although the strong model performance should still be interpreted cautiously and validated under the stricter time forward design.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

"Logistic Regression outperformed the frozen Week 4 baseline and achieved 1.00 Precision@20, showing that the learned model was superior."

### Revised claim

**Observed:** Logistic Regression achieved higher Precision@K than the frozen Week-4 baseline on the client grouped test split, including 1.00 Precision@20.

**Measured:** When the same modeling approach was evaluated with a stricter time forward design, Precision@20 was 0.85, Precision@50 was 0.86, and Precision@100 was 0.82. These values were lower than the Week 5 grouped split results but remained above the 30.75% test set base rate.

**Directional:** The results suggest that the learned model contains useful signal for ranking pages associated with future CTR decline, but the strength of the signal is lower under a future-period evaluation.

**Decision support:** The model can help prioritize pages for human review, but the evidence does not establish that the model will maintain the Week-5 level of performance on future periods or that a ranked page will necessarily benefit from a content refresh.

| Version | Claim |
|---|---|
| **Original W05 claim** | Logistic Regression outperformed the frozen Week-4 baseline and achieved 1.00 Precision@20, showing that the learned model was superior. |
| **Revised W06 claim** | **Observed:** Logistic Regression achieved higher Precision@K than the frozen Week-4 baseline on the Week-5 client-grouped test split. **Measured:** Under the stricter time-forward validation, Precision@20 was 0.85, Precision@50 was 0.86, and Precision@100 was 0.82, compared with a 30.75% test-set base rate. **Directional:** The results suggest useful signal for ranking pages associated with future CTR decline, but the strength of the signal was lower under future-period validation. **Decision support:** The model can help prioritize pages for human review, but the evidence does not show that it will maintain the Week-5 performance on future periods or that a ranked page will necessarily benefit from a content refresh. |

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.